In [18]:
# ================================================================
# STEP 9: EDA — 피처 분리 능력 분석
# ================================================================
from scipy import stats

if 'panel' not in dir():
    panel = pd.read_csv('./p_project_features.csv', encoding='utf-8-sig')
    snap  = pd.read_csv('./p_project_snapshot.csv', encoding='utf-8-sig')

# 9-1. 피처별 Mann-Whitney U 검정 (폐업 vs 생존)
#   alternative='greater': 폐업 점포의 리스크 피처값이 생존 점포보다 유의미하게 큰지 검증
results = []
for f in FEAT_ALL:
    dw = 'dw_' + f
    c  = snap[snap['is_closed_obs']==1][dw].dropna()
    a  = snap[snap['is_closed_obs']==0][dw].dropna()
    if len(c) < 3 or len(a) < 3:
        continue
    stat, pval = stats.mannwhitneyu(c, a, alternative='greater')
    effect = (2*stat)/(len(c)*len(a)) - 1  # rank-biserial correlation (-1~1)
    results.append({'feature':f,'closed_mean':c.mean(),'alive_mean':a.mean(),
                     'effect':effect,'pval':pval})

rdf = pd.DataFrame(results).sort_values('effect', ascending=False)
print('피처별 분리 능력 (effect 순위 = rank-biserial correlation, 클수록 더 유의미한 분리)')
print(f'  {"feature":<28} {"폐업avg":>8} {"생존avg":>8} {"effect":>7} {"p-val":>8}')
print('  ' + '-'*60)
for _, r in rdf.iterrows():
    sig = '*' if r.pval < 0.05 else ' '
    print(f'  {sig}{r["feature"]:<27} {r["closed_mean"]:>8.2f} {r["alive_mean"]:>8.2f} {r["effect"]:>7.3f} {r["pval"]:>8.4f}')

# 9-2. 이벤트 정렬: 폐업 전 매출 버킷 궤적 (Event Alignment)
# 핵심 인사이트: 추세 피처가 수준 피처보다 훨씬 강한 분리 능력 (effect ~0.4 vs ~0.17)
print('\n이벤트 정렬: 폐업 점포의 매출 버킷/업종순위 궤적')
print(f'  {"시점":<14} {"RC_M1_SAA":>10} {"순위%":>8} {"n":>5}')
cp = panel[panel['is_closed_obs']==1]
for t in [12,9,6,3,2,1,0]:
    rows = cp[cp['months_to_close'].between(t-0.5, t+0.5)]
    if len(rows) > 0:
        label = f'폐업 T-{t}' if t > 0 else '폐업 월T=0'
        print(f'  {label:<14} {rows["RC_M1_SAA"].mean():>10.2f} {rows["M12_SME_RY_SAA_PCE_RT"].mean():>8.1f} {len(rows):>5}')

# 생존 점포 (관측 종료 기준)
def ym_idx(ym): return (int(ym)//100)*12 + int(ym)%100
ap = panel[panel['is_closed_obs']==0].copy()
ap = ap[ap['TA_YM'].notna()]
ap['months_ago'] = ap['TA_YM'].apply(ym_idx) - ym_idx(202412)
print()
for t in [0,-3,-6,-12]:
    rows = ap[ap['months_ago'].between(t-0.5, t+0.5)]
    label = f'생존 현재T={t}'
    if len(rows) > 0:
        print(f'  {label:<14} {rows["RC_M1_SAA"].mean():>10.2f} {rows["M12_SME_RY_SAA_PCE_RT"].mean():>8.1f} {len(rows):>5}')

print('\n[KEY INSIGHT] 폐업 직전 매출 버킷 급등 + 추세 피처의 분리 능력이 수준 피처 2배 이상')


피처별 분리 능력 (effect 순위 = rank-biserial correlation, 클수록 더 유의미한 분리)
  feature                         폐업avg    생존avg  effect    p-val
  ------------------------------------------------------------
  *f_sales_trend                   0.16    -0.04   0.463   0.0000
  *f_vs_ind_trend                 11.60    -3.62   0.431   0.0000
  *f_trx_trend                     0.14    -0.03   0.341   0.0009
  *f_rank_ind_trend                0.51    -2.27   0.300   0.0031
  *f_rank_dist_trend               0.18    -1.41   0.287   0.0041
  *f_resid_ratio                 -25.93   -34.62   0.204   0.0267
  *f_spend_lvl                     3.87     3.50   0.187   0.0383
  *f_sales_lvl                     3.86     3.52   0.174   0.0496
   f_float_ratio                  59.12    52.75   0.168   0.0565
   f_vs_ind_sales                -93.25  -121.73   0.163   0.0612
   f_float_trend                   2.42     0.24   0.139   0.1198
   f_peer_close_dist               8.48     8.62   0.137   0.1499
   f_return_tr

In [19]:
# ================================================================
# STEP 10: Reference DB — 폐업 패턴 유사도 스코어
# ================================================================
# 폐업 전 6개월 패턴을 centroid로 요약,
# 각 점포의 현재 패턴과의 유사도를 추가 리스크 신호로 활용

# 10-1. 폐업 전 6개월 피처 centroid 계산
closed_pre = panel[
    (panel['is_closed_obs']==1) & (panel['months_to_close'].between(0,5))
].copy()
print(f'폐업 전 6개월 데이터: {len(closed_pre)}행, {closed_pre["ENCODED_MCT"].nunique()}개 폐업 점포')

feat_for_ref = [f for f in FEAT_ALL if closed_pre[f].notna().mean() > 0.8]
centroid     = closed_pre.groupby('ENCODED_MCT')[feat_for_ref].mean().mean()
feat_std_all = panel[feat_for_ref].std().replace(0, 1)  # 전체 데이터 std로 정규화

print(f'Reference 피처: {len(feat_for_ref)}개')
print('Centroid (폐업 전 6개월 평균):')
for f in ['f_sales_trend','f_trx_trend','f_vs_ind_trend','f_rank_ind_trend']:
    if f in centroid.index:
        print(f'  {f:<30}: {centroid[f]:+.3f}')

# 10-2. 각 점포의 DW 피처 → centroid까지 Mahalanobis 거리 (변동성 보정)
dw_cols = ['dw_'+f for f in feat_for_ref]
snap_ref = snap[dw_cols].copy()
snap_ref.columns = feat_for_ref

# 표준화 후 유클리드 거리 (= 변수별 가중치 균등)
snap_norm = (snap_ref - centroid) / feat_std_all
dist = np.sqrt((snap_norm.fillna(0)**2).sum(axis=1))
snap['pattern_dist']       = dist
snap['pattern_similarity'] = 1 - dist.rank(pct=True)  # 0~1, 높을수록 폐업 패턴과 유사

print()
print('pattern_similarity (업종 정규화 없이 원시 비교):')
print(snap.groupby('is_closed_obs')['pattern_dist'].describe().round(2))

# 10-3. pattern_similarity를 risk_score에 통합
snap['risk_score_v2'] = (
    0.45 * snap['score_internal'] +
    0.25 * snap['score_competitive'] +
    0.15 * snap['score_external'] +
    0.15 * snap['pattern_similarity'].fillna(0.5) * 100
)

c2 = snap[snap['is_closed_obs']==1]['risk_score_v2']
a2 = snap[snap['is_closed_obs']==0]['risk_score_v2']
print(f'\nrisk_score_v2: 폐업={c2.mean():.1f}, 생존={a2.mean():.1f}, 분리={c2.mean()>a2.mean()}')

snap.to_csv('./p_project_snapshot.csv', index=False, encoding='utf-8-sig')
print('[SAVED] p_project_snapshot.csv (pattern_similarity, risk_score_v2 추가)')


폐업 전 6개월 데이터: 165행, 30개 폐업 점포
Reference 피처: 16개
Centroid (폐업 전 6개월 평균):
  f_sales_trend                 : +0.190
  f_trx_trend                   : +0.155
  f_vs_ind_trend                : +13.794
  f_rank_ind_trend              : +0.642

pattern_similarity (업종 정규화 없이 원시 비교):
                count  mean   std   min   25%   50%   75%    max
is_closed_obs                                                   
0              4153.0  3.24  1.38  0.00  2.34  3.01  3.85  16.18
1                30.0  3.00  1.03  1.24  2.26  2.94  3.54   5.29

risk_score_v2: 폐업=58.8, 생존=50.3, 분리=True
[SAVED] p_project_snapshot.csv (pattern_similarity, risk_score_v2 추가)
